# 課程 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是構建 AI 代理的統一框架。它提供一個乾淨且可組合的架構，包含四個核心組件：

- **Client** – 連接到 AI 模型端點並處理通訊
- **Agent** – 將客戶端包裝在指令和工具定義中
- **Tools** – 透過模型可呼叫的自訂函數擴展代理能力
- **Session** – 維護多回合交互的對話歷史

在本課程中，我們將使用這些概念建立一個<strong>旅遊訂票代理</strong>，用以檢查目的地可用性。


## 設置


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## 了解 Agent 框架架構

Microsoft Agent 框架採用分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `FoundryChatClient` 連接到 Azure OpenAI 部署。它負責身份驗證、請求格式化和回應解析。
2. **Agent** – 透過 `provider.create_agent()` 從 client 創建的 agent，將模型存取與指令（系統提示）和工具結合起來。
3. **Tools** – 使用 `@tool` 裝飾的 Python 函數，agent 可以呼叫這些函數來執行動作或取得資料。
4. **Session** – 由 `agent.create_session()` 創建的 `AgentSession` 物件，用來存儲對話歷史，使 agent 能記住先前的上下文並進行多輪對話。

讓我們逐步建立每個層級。


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 使用 @tool 裝飾器新增工具

工具讓代理程式能執行超出產生文字的操作。`@tool` 裝飾器會將一般的 Python 函式轉換成代理程式可以呼叫的功能。

重要事項：
- 使用 `Annotated[type, "description"]`，讓模型理解每個參數。
- 函式說明（docstring）會成為模型所看到的工具描述。
- `approval_mode="never_require"` 表示該工具會自動執行，無需使用者確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 建立具備工具功能的代理人

現在我們將客戶端、指示和工具結合成一個代理人。`instructions` 作為系統提示 — 它們定義代理人的角色和行為。


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 使用會話的多輪對話

`AgentSession`（透過 `agent.create_session()` 建立）會追蹤對話中的所有訊息。透過在每次呼叫 `agent.run()` 時傳入相同的會話，代理人可以存取完整的對話歷史並回顧先前的訊息。

我們傳入 `tools=[check_destination_availability]`，讓代理人在每回合都能呼叫我們的可用性檢查器。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 摘要

在本課程中，您探討了 Microsoft Agent Framework 的四大支柱：

| 概念 | 您所學習的內容 |
|---------|------------------|
| <strong>客戶端</strong> | `FoundryChatClient` 使用基於憑證的身份驗證連接到 Azure OpenAI |
| <strong>代理人</strong> | `provider.create_agent()` 將模型連結與指令及名稱綁定 |
| <strong>工具</strong> | `@tool` 裝飾器公開 Python 函數供代理人調用 |
| <strong>會話</strong> | `agent.create_session()` 在多輪對話中維持對話歷史 |

這些基礎構件相互組合，創建能夠進行自然對話、調用外部函數並維持上下文的代理人 —— 為後續課程中更進階的代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
